In [13]:
import numpy as np
import pandas as pd

In [14]:
# Set seed so your demo produces the exact same numbers every time
np.random.seed(42)
n_samples = 100

# Create 85 Normal Transactions (Class = 0)
normal_data = {
    'Amount_INR': np.random.randint(150, 15000, size=85),
    'Hour_of_Day': np.random.randint(8, 23, size=85),          # Daytime/Evening (8 AM - 10 PM)
    'V1_Device_Risk': np.random.normal(-0.5, 0.8, size=85).round(2),
    'V2_Velocity_Score': np.random.normal(-0.4, 0.7, size=85).round(2),
    'Is_International': np.random.choice([0, 1], size=85, p=[0.95, 0.05]),
    'Class': np.zeros(85, dtype=int)
}

Amount_INR: Transaction value(e.g., ₹450 for groceries vs. ₹85,000 for late-night electronics).
Hour_of_Day: 0 to 23 (Fraudulent transactions spike between 1 AM and 4 AM).
V1_Device_Risk: Anonymized PCA behavioral score for device/IP change (-2.0 = familiar phone, +3.5 = brand-new device/VPN).
V2_Velocity_Score: Anonymized PCA score for rapid consecutive swipes/UPI attempts (-1.5 = normal spacing, +3.0 = 5 attempts in 2 minutes).
Is_International: 0 = Domestic (India), 1 = International gateway.
Class (Target Label): 0 = Genuine Transaction, 1 = Fraudulent Transaction.

In [15]:
## Create 15 Fraudulent Transactions (Class = 1)
fraud_data = {
    'Amount_INR': np.random.randint(25000, 95000, size=15),
    'Hour_of_Day': np.random.choice([1, 2, 3, 4, 23], size=15), # Late night/early morning
    'V1_Device_Risk': np.random.normal(2.2, 0.9, size=15).round(2),
    'V2_Velocity_Score': np.random.normal(1.9, 0.8, size=15).round(2),
    'Is_International': np.random.choice([0, 1], size=15, p=[0.40, 0.60]),
    'Class': np.ones(15, dtype=int)
}

In [16]:
# Combine and shuffle the dataset
df = pd.concat([pd.DataFrame(normal_data), pd.DataFrame(fraud_data)], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [17]:
df.head()

,Amount_INR,Hour_of_Day,V1_Device_Risk,V2_Velocity_Score,Is_International,Class
0,3223,12,1.53,-0.02,0,0
1,8988,15,-0.51,0.21,0,0
2,5636,14,0.02,-1.45,0,0
3,4808,8,-0.85,0.10,0,0
4,3155,15,-0.42,-0.76,0,0


In [18]:
# Introduce 2 realistic "messy" missing values in Amount_INR and clean them
df.loc[4, 'Amount_INR'] = np.nan
df['Amount_INR'] = df['Amount_INR'].fillna(df['Amount_INR'].median())

In [19]:
print("✅ Dataset Loaded Successfully! Total Transactions:", len(df))
df.head(8)

✅ Dataset Loaded Successfully! Total Transactions: 100


,Amount_INR,Hour_of_Day,V1_Device_Risk,V2_Velocity_Score,Is_International,Class
0,3223.0,12,1.53,-0.02,0,0
1,8988.0,15,-0.51,0.21,0,0
2,5636.0,14,0.02,-1.45,0,0
3,4808.0,8,-0.85,0.10,0,0
4,8583.0,15,-0.42,-0.76,0,0
5,2897.0,20,0.62,-0.72,0,0
6,7099.0,21,0.60,-0.30,0,0
7,1735.0,10,0.34,-0.63,0,0


In [20]:
# Check how many Normal (0) vs Fraud (1) transactions exist
counts = df['Class'].value_counts()
print("Transaction Breakdown:")
print(f"0 (Genuine): {counts[0]} transactions ({counts[0]}%)")
print(f"1 (Fraud)  : {counts[1]} transactions ({counts[1]}%)")

Transaction Breakdown:
0 (Genuine): 85 transactions (85%)
1 (Fraud)  : 15 transactions (15%)


Accuracy = Total Correct Predictions/Total Transactions = 85(correct)+0(fraud)/100 = 85/100 = 85%On paper, an 85% accuracy score sounds impressive. But in reality, the model caught zero fraudsters .

In [8]:
from sklearn.model_selection import train_test_split

# X = Features (The clues), y = Label (The answer)
X = df.drop(columns=['Class'])
y = df['Class']

# Split: 80% Training Data, 20% Testing Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Total Rows: {len(df)}")
print(f"Training Set (Study Material): {X_train.shape[0]} rows")
print(f"Testing Set  (Unseen Exam)   : {X_test.shape[0]} rows")

Total Rows: 100
Training Set (Study Material): 80 rows
Testing Set  (Unseen Exam)   : 20 rows


In [9]:
from sklearn.linear_model import LogisticRegression

# 1. Initialize and Train the Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("🎯 Model Training Complete!")

# 2. Predict on the unseen 20% test data
predictions = model.predict(X_test)

# 3. Create a clean table of the first 10 test cases WITHOUT actual answers
demo_df = X_test.iloc[:10].copy().reset_index(drop=True)
demo_df['Model_Prediction'] = predictions[:10]

print("\n--- FIRST 10 UNSEEN TEST TRANSACTIONS (ACTUAL LABEL HIDDEN) ---")
demo_df

🎯 Model Training Complete!

--- FIRST 10 UNSEEN TEST TRANSACTIONS (ACTUAL LABEL HIDDEN) ---


,Amount_INR,Hour_of_Day,V1_Device_Risk,V2_Velocity_Score,Is_International,Model_Prediction
0,6570.0,9,-0.71,-0.57,0,0
1,5341.0,13,2.20,-1.01,0,0
2,7705.0,10,0.59,-0.31,0,0
3,85692.0,1,1.65,2.39,0,1
4,8988.0,15,-0.51,0.21,0,0
5,339.0,14,-1.36,-0.64,0,0
6,4705.0,12,0.52,-0.02,0,0
7,42955.0,1,1.12,2.82,1,1
8,1010.0,13,0.14,-1.41,0,0
9,10777.0,12,-1.14,-0.82,0,0


In [10]:
# Reveal the ground truth alongside the model's prediction
reveal_df = demo_df[['Amount_INR', 'Hour_of_Day', 'Is_International', 'Model_Prediction']].copy()
reveal_df['Actual_Label'] = y_test.iloc[:10].values

# Helper column to label each outcome clearly
def classify_result(row):
    if row['Actual_Label'] == 1 and row['Model_Prediction'] == 1:
        return "✅ True Positive (Caught Fraud!)"
    elif row['Actual_Label'] == 0 and row['Model_Prediction'] == 0:
        return "✅ True Negative (Cleared Genuine)"
    elif row['Actual_Label'] == 0 and row['Model_Prediction'] == 1:
        return "⚠️ False Positive (False Alarm - Blocked Genuine Citizen)"
    else:
        return "🚨 False Negative (Missed Fraud!)"

reveal_df['Outcome_Type'] = reveal_df.apply(classify_result, axis=1)
reveal_df

,Amount_INR,Hour_of_Day,Is_International,Model_Prediction,Actual_Label,Outcome_Type
0,6570.0,9,0,0,0,✅ True Negative (Cleared Genuine)
1,5341.0,13,0,0,0,✅ True Negative (Cleared Genuine)
2,7705.0,10,0,0,0,✅ True Negative (Cleared Genuine)
3,85692.0,1,0,1,1,✅ True Positive (Caught Fraud!)
4,8988.0,15,0,0,0,✅ True Negative (Cleared Genuine)
5,339.0,14,0,0,0,✅ True Negative (Cleared Genuine)
6,4705.0,12,0,0,0,✅ True Negative (Cleared Genuine)
7,42955.0,1,1,1,1,✅ True Positive (Caught Fraud!)
8,1010.0,13,0,0,0,✅ True Negative (Cleared Genuine)
9,10777.0,12,0,0,0,✅ True Negative (Cleared Genuine)


False Positive (Predicted 1, Actual 0): Imagine a person is trying to pay ₹12,000 for a hospital bill or railway ticket and our model falsely flags it as fraud and blocks their card. What is the real-world impact? 

False Negative (Predicted 0, Actual 1): Imagine a fraudster drains ₹75,000 at 2:00 AM from a new device and our model predicts 0 (Normal). What happens?" .

Takeaway: In e-Governance and banking systems, Accuracy alone is never enough—officers must decide whether it is worse to trigger a false alarm or miss an actual threat.